# Session 12: Bash Scripting for ML Workflows — Open in colab!
## file >> open in colab

*"Automating the boring parts so you can focus on the interesting ones"*

In Session 11, we learned to navigate the filesystem, manipulate files, and run Python from the command line. Now we learn to write **scripts** -- sequences of commands saved to a file and executed as a program. This is how real ML pipelines work: a training script that takes hyperparameters as arguments, logs its output, and can be rerun with different settings.

---

## 1. Your First Bash Script

A script is just a text file containing commands that the shell executes in sequence. Instead of typing commands one at a time, we write them into a file and run the file. This matters because ML workflows involve the same sequence of steps repeated with different settings -- and repeating things manually is how mistakes happen.

We use the `%%writefile` magic to create script files directly from notebook cells.

In [ ]:
%%writefile hello.sh
#!/bin/bash
echo "Hello from a Bash script!"
echo "Today is $(date)"
echo "Running on $(uname -s)"

The first line, `#!/bin/bash`, is called a **shebang**. It tells the operating system which interpreter to use when running the file. Without it, the system would not know whether this is a Bash script, a Python script, or something else entirely.

## Before we go into shell scripts in-depth, let's try a basic orientation activity with this script:

**You will need to run the above cells first!** 

Remember with vi: 

1. you have to press i to start editing
2. press esc key when done
3. press :wq  to save and quit!

Open your terminal in colab and you should see:

```bash
/content#

```

### First let's get oriented

```bash
before running cells above ls shows:
/content# ls
sample_data
/content# pwd
/content
```
```bash
after running the first cell:
/content# ls
hello.sh  sample_data
/content# cat hello.sh
#!/bin/bash
echo "Hello from a Bash script!"
echo "Today is $(date)"
echo "Running on $(uname -s)"
```

#### We can run any of these lines in our shell script interactively

```bash
/content# cat hello.sh
#!/bin/bash
echo "Hello from a Bash script!"
echo "Today is $(date)"
echo "Running on $(uname -s)"
/content# echo "Today is $(date)"
#!/bin/bash
Today is Tue Apr 28 01:30:54 AM UTC 2026
/content# echo "Running on $(uname -s)"
Running on Linux
```

#### or we can run the whole script:
```bash
/content# ls
hello.sh  sample_data
/content# ./hello.sh
-bash: ./hello.sh: Permission denied
```

#### Oh why did that happen?


Think back to Session 11.

Before we can run the script, we need to make it executable with `chmod u+x`. We covered permissions in Session 11 -- the `+x` flag grants execute permission, u grants this only to the user that owns the file (you!).

```bash
/content# ls -l
total 8
-rw-r--r-- 1 root root   99 Apr 28 01:26 hello.sh
drwxr-xr-x 1 root root 4096 Apr 16 13:28 sample_data
/content# chmod u+x ./hello.sh
/content# ls -l
total 8
-rwxr--r-- 1 root root   99 Apr 28 01:26 hello.sh
drwxr-xr-x 1 root root 4096 Apr 16 13:28 sample_data
/content# ./hello.sh 
Hello from a Bash script!
Today is Tue Apr 28 01:33:01 AM UTC 2026
Running on Linux
```

This script has a tiny problem with it, when running bash scripts we might prefer if they used our own environment!

What is our environment?

try running:

```bash
/content# /bin/env bash --version
GNU bash, version 5.1.16(1)-release (x86_64-pc-linux-gnu)
Copyright (C) 2020 Free Software Foundation, Inc.
License GPLv3+: GNU GPL version 3 or later <http://gnu.org/licenses/gpl.html>

This is free software; you are free to change and redistribute it.
There is NO WARRANTY, to the extent permitted by law.

/content# /bin/bash --version
GNU bash, version 5.1.16(1)-release (x86_64-pc-linux-gnu)
Copyright (C) 2020 Free Software Foundation, Inc.
License GPLv3+: GNU GPL version 3 or later <http://gnu.org/licenses/gpl.html>

This is free software; you are free to change and redistribute it.
There is NO WARRANTY, to the extent permitted by law.
```
On this system there is no mismatch, but that might not be the case on every system!

#### we can see our current shell environment like this

```
/content# env

TERM=screen
LESSOPEN=| /usr/bin/lesspipe %s
TMUX_PANE=%0
SHLVL=1
COLAB_LANGUAGE_SERVER_PROXY=/usr/colab/bin/language_service
UV_CONSTRAINT=
PYTHONUTF8=1
COLAB_GPU=
GCS_READ_CACHE_BLOCK_SIZE_MB=16
LC_ALL=en_US.UTF-8
COLAB_FILE_HANDLER_ADDR=localhost:3453
PATH=/opt/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin
COLAB_IMAGE_TYPE=cpu
COLAB_DEBUG_ADAPTER_MUX_PATH=/usr/local/bin/dap_multiplexer
PYTHONWARNINGS=ignore:::pip._internal.cli.base_command
print("Hello from python")
DEBIAN_FRONTEND=noninteractive
COLAB_BACKEND_VERSION=next
OLDPWD=/
TERM_PROGRAM=tmux
_=/usr/bin/env

```

The PATH environment variable here is the most important! It tells bash how to find your commands (most commands are actually just files on the filesystem somewhere!) — some are built-in to bash (just like python built-in libraries like math). Bash uses PATH (just a list of file paths seperated by :) as a list of locations to search for your commands in order of preference first to last.

#### Let's check if we have python now

```bash
/content# python --version
Python 3.12.13
/content# which python
/usr/local/bin/python
```

#### we do!

Let's open up that hello.sh in vi and add `python ./my_script.py`

```bash
/content# vi ./hello.sh
/content# cat ./hello.sh 
#!/bin/bash
echo "Hello from the Bash script!"
echo "Today is $(date)"
echo "Running on $(uname -s)"

python ./my_script.py


```

#### Let's try and check if that works interactively

```bash
/content# python ./my_script.py
python3: can't open file '/content/./my_script.py': [Errno 2] No such file or directory
```

### Oops we need to make that file! Let's use vi! remember: i to insert, esc to switch back, :wq
```bash
/content# vi ./my_script.py
/content# cat ./my_script.py
print("Hello from python")
```

#### let's run our python script
```bash
/content# python ./my_script.py
Hello from python
```

#### we can now run the same script from our bash script like this:
```bash
/content# ./hello.sh 
Hello from the Bash script!
Today is Tue Apr 28 01:43:38 AM UTC 2026
Running on Linux
Hello from python
```

### Variables

Variables in Bash store values for reuse. The syntax is `NAME="value"` -- note there must be **no spaces** around the `=` sign. This is a common source of errors: `NAME = "value"` will fail because Bash interprets `NAME` as a command.

We reference a variable's value with `$NAME` or `${NAME}`.

In [ ]:
%%writefile greet.sh
#!/bin/bash
NAME="world"
echo "Hello, $NAME!"
echo "This script is called: $0"

In [ ]:
!chmod +x greet.sh
!./greet.sh

### Command-line arguments

Scripts become truly useful when they accept arguments -- values passed in when you run the script. Bash provides special variables for this:

| Variable | Meaning |
|---|---|
| `$0` | The script's own name |
| `$1`, `$2`, ... | The first, second, ... arguments |
| `$#` | The number of arguments |
| `$@` | All arguments as a list |

In [ ]:
%%writefile greet_user.sh
#!/bin/bash
echo "Script: $0"
echo "Number of arguments: $#"
echo "All arguments: $@"
echo "Hello, $1! Welcome to session $2."

In [ ]:
!chmod +x greet_user.sh
!./greet_user.sh Alice 12

### Exercise 1: Greet by name

**Try this:** Write a script called `personal_greet.sh` that takes a person's name as the first argument and their favourite model as the second. It should output:

```
Hello, Alice!
Your favourite model is ResNet.
```

If no arguments are provided, it should use default values.

In [ ]:
%%writefile personal_greet.sh
#!/bin/bash
NAME=${1:-"Stranger"}
MODEL=${2:-"a neural network"}
echo "Hello, $NAME!"
echo "Your favourite model is $MODEL."

In [ ]:
!chmod +x personal_greet.sh
!echo "--- With arguments ---"
!./personal_greet.sh Alice ResNet
!echo ""
!echo "--- Without arguments ---"
!./personal_greet.sh

---

## 2. Variables and Arithmetic

ML scripts need to work with numbers -- epochs, batch sizes, learning rates -- and build paths dynamically. Bash handles string variables naturally but requires specific syntax for arithmetic.

### String variables

We can use variables to build up configuration values that get reused throughout a script.

In [ ]:
%%writefile config_demo.sh
#!/bin/bash
MODEL="resnet"
DATASET="cifar10"
EXPERIMENT="${MODEL}_${DATASET}"

echo "Model:      $MODEL"
echo "Dataset:    $DATASET"
echo "Experiment: $EXPERIMENT"

In [ ]:
!chmod +x config_demo.sh
!./config_demo.sh

### Integer arithmetic

Bash performs integer arithmetic inside `$(( ))`. This is useful for calculating total steps, batch counts, or epoch indices.

In [ ]:
%%writefile arithmetic_demo.sh
#!/bin/bash
EPOCHS=10
BATCHES=100
TOTAL=$((EPOCHS * BATCHES))

echo "Epochs:  $EPOCHS"
echo "Batches: $BATCHES"
echo "Total training steps: $TOTAL"

WARMUP=$((TOTAL / 10))
echo "Warmup steps (10%): $WARMUP"

In [ ]:
!chmod +x arithmetic_demo.sh
!./arithmetic_demo.sh

### Command substitution

Command substitution captures the output of a command and stores it in a variable. The syntax is `$(command)`. This is how scripts incorporate dynamic information -- timestamps, file counts, system details -- into their logic.

In [ ]:
%%writefile substitution_demo.sh
#!/bin/bash
DATE=$(date +%Y-%m-%d)
TIME=$(date +%H%M%S)
HOSTNAME=$(hostname)

echo "Date: $DATE"
echo "Time: $TIME"
echo "Host: $HOSTNAME"

# Build a path dynamically
MODEL="resnet"
DATASET="cifar10"
OUTPUT_DIR="runs/${MODEL}_${DATASET}_${DATE}"
echo "Output directory: $OUTPUT_DIR"

In [ ]:
!chmod +x substitution_demo.sh
!./substitution_demo.sh

### Exercise 2: Timestamped experiment directory

**Try this:** Write a script called `setup_experiment.sh` that:

1. Takes a model name as an argument (default: `"baseline"`)
2. Creates a timestamped directory under `/tmp/experiments/`
3. Prints the path it created

In [ ]:
%%writefile setup_experiment.sh
#!/bin/bash
MODEL=${1:-"baseline"}
TIMESTAMP=$(date +%Y%m%d_%H%M%S)
EXPERIMENT_DIR="/tmp/experiments/${MODEL}_${TIMESTAMP}"

mkdir -p "$EXPERIMENT_DIR"
echo "Created experiment directory: $EXPERIMENT_DIR"
ls -la /tmp/experiments/

In [ ]:
!chmod +x setup_experiment.sh
!./setup_experiment.sh resnet50

---

## 3. Conditionals -- Making Decisions

Scripts need to make decisions: does the dataset exist before we start training? Did the previous step succeed? Is the GPU available? Bash uses `if` statements with test expressions inside `[ ]` brackets.

### File tests

The most common conditionals in ML scripts check whether files and directories exist.

| Test | Meaning |
|---|---|
| `[ -f file ]` | True if file exists and is a regular file |
| `[ -d dir ]` | True if directory exists |
| `[ -s file ]` | True if file exists and is non-empty |
| `[ ! -f file ]` | True if file does **not** exist |

In [ ]:
%%writefile check_files.sh
#!/bin/bash

# Check if a dataset exists before training
DATA_FILE="/tmp/experiments/train.csv"

if [ -f "$DATA_FILE" ]; then
    echo "Dataset found: $DATA_FILE"
    echo "Proceeding with training..."
else
    echo "Dataset not found: $DATA_FILE"
    echo "Creating sample dataset..."
    mkdir -p /tmp/experiments
    echo "id,feature,label" > "$DATA_FILE"
    echo "1,0.5,0" >> "$DATA_FILE"
    echo "2,0.8,1" >> "$DATA_FILE"
    echo "Dataset created."
fi

In [ ]:
!chmod +x check_files.sh
!echo "--- First run (file does not exist) ---"
!./check_files.sh
!echo ""
!echo "--- Second run (file now exists) ---"
!./check_files.sh

### String and numeric comparisons

Beyond file tests, we often compare variable values:

| Test | Meaning |
|---|---|
| `[ "$VAR" = "value" ]` | String equality |
| `[ "$VAR" != "value" ]` | String inequality |
| `[ $NUM -gt 5 ]` | Greater than |
| `[ $NUM -lt 5 ]` | Less than |
| `[ $NUM -eq 5 ]` | Equal to |
| `[ $NUM -ge 5 ]` | Greater than or equal |
| `[ $NUM -le 5 ]` | Less than or equal |

In [ ]:
%%writefile check_args.sh
#!/bin/bash
OPTIMIZER=${1:-"sgd"}
EPOCHS=${2:-5}

if [ "$OPTIMIZER" = "adam" ]; then
    echo "Using Adam optimiser (adaptive learning rate)"
elif [ "$OPTIMIZER" = "sgd" ]; then
    echo "Using SGD (stochastic gradient descent)"
else
    echo "Unknown optimiser: $OPTIMIZER"
    echo "Supported: sgd, adam"
    exit 1
fi

if [ $EPOCHS -gt 100 ]; then
    echo "Warning: $EPOCHS epochs is a lot. Are you sure?"
elif [ $EPOCHS -lt 1 ]; then
    echo "Error: need at least 1 epoch"
    exit 1
else
    echo "Training for $EPOCHS epochs."
fi

In [ ]:
!chmod +x check_args.sh
!./check_args.sh adam 10
!echo ""
!./check_args.sh sgd 200

### A practical pattern: pre-flight checks

Real training scripts should verify that everything is in place before starting a potentially long run. This is the ML equivalent of a pilot's pre-flight checklist.

In [ ]:
%%writefile preflight.sh
#!/bin/bash
echo "=== Pre-flight checks ==="

# Check Python is available
if ! command -v python &> /dev/null; then
    echo "FAIL: Python not found"
    exit 1
fi
echo "OK: Python found at $(which python)"

# Check dataset
if [ ! -f "/tmp/experiments/train.csv" ]; then
    echo "FAIL: Training data not found"
    exit 1
fi
echo "OK: Training data exists"

# Check output directory
OUTPUT_DIR="/tmp/experiments/results"
if [ ! -d "$OUTPUT_DIR" ]; then
    echo "INFO: Creating output directory: $OUTPUT_DIR"
    mkdir -p "$OUTPUT_DIR"
fi
echo "OK: Output directory ready"

echo ""
echo "All checks passed. Ready to train."

In [ ]:
!chmod +x preflight.sh
!./preflight.sh

### Exercise 3: Pre-training checks

**Try this:** Write a script called `check_ready.sh` that:

1. Checks whether a file called `config.yaml` exists in the current directory
2. Checks whether a `data/` directory exists
3. Prints a pass/fail summary
4. Exits with code 1 if any check fails, code 0 if all pass

In [ ]:
# Create the files so the check passes
!mkdir -p data
!echo "lr: 0.001" > config.yaml

In [ ]:
%%writefile check_ready.sh
#!/bin/bash
READY=true

echo "=== Readiness check ==="

if [ -f "config.yaml" ]; then
    echo "OK: config.yaml found"
else
    echo "FAIL: config.yaml not found"
    READY=false
fi

if [ -d "data" ]; then
    echo "OK: data/ directory found"
else
    echo "FAIL: data/ directory not found"
    READY=false
fi

echo ""
if [ "$READY" = true ]; then
    echo "All checks passed."
    exit 0
else
    echo "Some checks failed. Fix the issues above before training."
    exit 1
fi

In [ ]:
!chmod +x check_ready.sh
!./check_ready.sh

---

## 4. Loops -- Repetition

Loops are where scripting starts to save serious time. Instead of running a command once, we run it for every file in a directory, every learning rate in a list, or every epoch in a training run.

### `for` loops

The `for` loop iterates over a list of values. The syntax is:

```bash
for VARIABLE in value1 value2 value3; do
    commands using $VARIABLE
done
```

In [ ]:
%%writefile loop_demo.sh
#!/bin/bash

# Iterate over a list of models
echo "=== Models ==="
for MODEL in resnet vgg transformer; do
    echo "Model: $MODEL"
done

echo ""

# Iterate over files
echo "=== Shell scripts in current directory ==="
for FILE in *.sh; do
    echo "Found: $FILE"
done

In [ ]:
!chmod +x loop_demo.sh
!./loop_demo.sh

### Iterating over values -- the hyperparameter use case

This is where loops become genuinely powerful for ML. We can try different learning rates, optimisers, or batch sizes by simply listing the values we want to test.

In [ ]:
%%writefile lr_sweep.sh
#!/bin/bash
echo "=== Learning rate sweep ==="

for LR in 0.001 0.01 0.1; do
    echo "Training with lr=$LR ..."
    # In a real script, this would call python train.py --lr $LR
    echo "  (simulated result for lr=$LR)"
done

In [ ]:
!chmod +x lr_sweep.sh
!./lr_sweep.sh

### `while` loops

A `while` loop repeats as long as a condition is true. This is useful for counting epochs or waiting for a process to finish.

In [ ]:
%%writefile while_demo.sh
#!/bin/bash
EPOCH=1
MAX_EPOCHS=5

while [ $EPOCH -le $MAX_EPOCHS ]; do
    echo "Epoch $EPOCH/$MAX_EPOCHS: training..."
    EPOCH=$((EPOCH + 1))
done

echo "Training complete."

In [ ]:
!chmod +x while_demo.sh
!./while_demo.sh

### Nested loops -- the hyperparameter sweep

The killer use case for loops in ML is the **hyperparameter sweep**: trying every combination of settings. With nested `for` loops, we can test 3 learning rates with 2 optimisers -- 6 experiments -- in a few lines of code.

In [ ]:
%%writefile nested_sweep.sh
#!/bin/bash
echo "=== Hyperparameter sweep ==="
echo ""

COUNT=0
for LR in 0.001 0.01 0.1; do
    for OPT in sgd adam; do
        COUNT=$((COUNT + 1))
        echo "Run $COUNT: lr=$LR, optimizer=$OPT"
    done
done

echo ""
echo "Total runs: $COUNT"

In [ ]:
!chmod +x nested_sweep.sh
!./nested_sweep.sh

### Exercise 4: Loop over learning rates

**Try this:** Write a script that iterates over learning rates `0.0001 0.001 0.01 0.1` and prints each one along with its index number (Run 1, Run 2, etc.).

In [ ]:
%%writefile lr_loop.sh
#!/bin/bash
RUN=1
for LR in 0.0001 0.001 0.01 0.1; do
    echo "Run $RUN: learning_rate=$LR"
    RUN=$((RUN + 1))
done

In [ ]:
!chmod +x lr_loop.sh
!./lr_loop.sh

---

## 5. A Real Training Script

Now we bring everything together. We will create a mock Python training script and a Bash wrapper that manages arguments, output directories, and logging. This is the pattern used in real ML workflows -- even when the training code is complex, the orchestration is Bash.

### The Python training script

First, we create a `train.py` that simulates training. It accepts hyperparameters via `argparse` and prints epoch-by-epoch metrics. No PyTorch required -- we simulate the loss curve with simple arithmetic.

In [ ]:
%%writefile train.py
import argparse
import time
import random
import os
import json

parser = argparse.ArgumentParser(description="Mock training script")
parser.add_argument("--lr", type=float, default=0.001, help="Learning rate")
parser.add_argument("--epochs", type=int, default=5, help="Number of epochs")
parser.add_argument("--optimizer", choices=["sgd", "adam"], default="sgd", help="Optimiser")
parser.add_argument("--output-dir", type=str, default="results", help="Output directory")
args = parser.parse_args()

# Seed for reproducibility within each run
random.seed(42)

print(f"Configuration: lr={args.lr}, optimizer={args.optimizer}, epochs={args.epochs}")
print(f"Output directory: {args.output_dir}")
print()

# Simulate training
results = []
for epoch in range(1, args.epochs + 1):
    # Simulate a loss curve that depends on learning rate and optimiser
    opt_factor = 1.2 if args.optimizer == "adam" else 1.0
    loss = 2.3 / (epoch * args.lr * 100 * opt_factor + 1) + random.uniform(-0.05, 0.05)
    acc = min(0.95, 0.1 + epoch * 0.08 * opt_factor + random.uniform(-0.02, 0.02))
    print(f"Epoch {epoch}/{args.epochs} | loss: {loss:.4f} | accuracy: {acc:.4f}")
    results.append({"epoch": epoch, "loss": round(loss, 4), "accuracy": round(acc, 4)})
    time.sleep(0.1)

# Save results to output directory
os.makedirs(args.output_dir, exist_ok=True)
results_file = os.path.join(args.output_dir, "metrics.json")
with open(results_file, "w") as f:
    json.dump({"config": vars(args), "results": results}, f, indent=2)

print(f"\nTraining complete. Results saved to {results_file}")

Let us verify that the training script works on its own.

In [ ]:
!python train.py --lr 0.01 --epochs 3 --optimizer adam --output-dir /tmp/test_run

### The Bash wrapper script

Now we write `run_experiment.sh` -- a Bash script that wraps the Python training script. It handles:

- **Default arguments** with `${1:-default}` syntax
- **Timestamped output directories** so runs never overwrite each other
- **Logging** with `tee`, which writes to both the screen and a file simultaneously
- **Error capture** with `2>&1`, which redirects stderr to stdout so error messages also appear in the log

In [ ]:
%%writefile run_experiment.sh
#!/bin/bash
LR=${1:-0.001}
EPOCHS=${2:-5}
OPT=${3:-sgd}
TIMESTAMP=$(date +%Y%m%d_%H%M%S)
OUTPUT_DIR="results/${OPT}_lr${LR}_${TIMESTAMP}"

mkdir -p "$OUTPUT_DIR"

echo "=============================="
echo "Starting training"
echo "  Learning rate: $LR"
echo "  Epochs:        $EPOCHS"
echo "  Optimiser:     $OPT"
echo "  Output:        $OUTPUT_DIR"
echo "=============================="
echo ""

python train.py --lr "$LR" --epochs "$EPOCHS" --optimizer "$OPT" \
    --output-dir "$OUTPUT_DIR" 2>&1 | tee "$OUTPUT_DIR/training.log"

echo ""
echo "Training complete. Log saved to $OUTPUT_DIR/training.log"

Two new concepts here:

- **`tee`** is named after a T-junction in plumbing. It takes input and splits it into two streams: one goes to the screen (stdout) and one goes to a file. This way you can watch training progress in real time *and* keep a log for later.
- **`2>&1`** redirects file descriptor 2 (stderr, where error messages go) to file descriptor 1 (stdout). This ensures that any error messages from Python also end up in the log file, not just on the screen.

In [ ]:
!chmod +x run_experiment.sh
!./run_experiment.sh 0.01 5 adam

Let us verify the log file was created and contains the training output.

In [ ]:
!echo "=== Files in results/ ==="
!ls -R results/
!echo ""
!echo "=== Log file contents ==="
!cat results/*/training.log 2>/dev/null | head -10

---

## 6. Output Redirection and Logging

Logging is not optional in ML -- it is how you debug failed runs, compare results, and prove to your future self what happened. This section covers the practical patterns for capturing and reviewing output.

### `>` vs `>>` recap

We covered these in Session 11, but they are worth reinforcing:

- `>` **overwrites** the file (creates it if it does not exist)
- `>>` **appends** to the file (creates it if it does not exist)

Getting these confused is how you lose yesterday's results.

In [ ]:
# Overwrite: only the second line survives
!echo "Run 1: acc=0.85" > /tmp/experiments/log.txt
!echo "Run 2: acc=0.87" > /tmp/experiments/log.txt
!echo "Overwrite (>) result:"
!cat /tmp/experiments/log.txt

!echo ""

# Append: both lines survive
!echo "Run 1: acc=0.85" > /tmp/experiments/log2.txt
!echo "Run 2: acc=0.87" >> /tmp/experiments/log2.txt
!echo "Append (>>) result:"
!cat /tmp/experiments/log2.txt

### Logging everything from a training run

The most common pattern is to redirect all output (both stdout and stderr) to a log file. This captures everything -- including error messages and stack traces.

In [ ]:
# Run training with all output going to a log file
!python train.py --lr 0.001 --epochs 3 --output-dir /tmp/experiments/logged_run > /tmp/experiments/training.log 2>&1

!echo "Training finished. Nothing appeared on screen because it all went to the log."
!echo ""
!echo "=== Contents of training.log ==="
!cat /tmp/experiments/training.log

### `tail -f` -- watching a log in real time

In a terminal (not a notebook), `tail -f logfile` follows the log as new lines are written. This is how you monitor a training run:

```bash
# In terminal 1: start training
python train.py --epochs 100 > training.log 2>&1 &

# In terminal 2: watch the log
tail -f training.log
```

Press `Ctrl+C` to stop following (the training continues). We cannot demonstrate `tail -f` in a notebook because it blocks indefinitely, but here is the non-following version:

In [ ]:
# tail without -f shows the last N lines (default 10)
!echo "=== Last 3 lines of the log ==="
!tail -3 /tmp/experiments/training.log

### Exercise 5: Log to a file

**Try this:** Run `train.py` with `--lr 0.1 --epochs 5` and redirect all output to `/tmp/experiments/exercise5.log`. Then use `cat` to verify the log contents and `tail -1` to see the final line.

In [ ]:
!python train.py --lr 0.1 --epochs 5 --output-dir /tmp/experiments/ex5 > /tmp/experiments/exercise5.log 2>&1
!echo "=== Full log ==="
!cat /tmp/experiments/exercise5.log
!echo ""
!echo "=== Final line ==="
!tail -1 /tmp/experiments/exercise5.log

---

## 7. Background Processes and Long Jobs

Training runs can take hours or days. You need to start them and walk away -- or start several and let them run in parallel. This section covers the tools for managing long-running processes.

### `&` -- run in background

Adding `&` to the end of a command runs it in the background, freeing your terminal for other work. The shell prints the process ID (PID) so you can check on it or stop it later.

In [ ]:
# Background processes: in a terminal you would run:
#   python train.py --lr 0.01 --epochs 3 > bg.log 2>&1 &
#   echo $!   # prints the process ID
#
# The & at the end sends the job to the background.
# This does not work in notebook cells — use the Colab terminal instead.
# Here we demonstrate the concept by running synchronously:

!python train.py --lr 0.01 --epochs 3 --output-dir /tmp/experiments/bg_run > /tmp/experiments/bg.log 2>&1
!echo "Job complete. Log contents:"
!cat /tmp/experiments/bg.log

### `nohup` -- survive terminal disconnect

If you close your terminal (or your SSH connection drops), background processes normally die. `nohup` ("no hang up") prevents this -- the process keeps running even after you disconnect.

This is essential for remote training on cloud machines or university clusters:

```bash
nohup python train.py --epochs 100 > training.log 2>&1 &
echo $!   # print the process ID
```

The `$!` variable holds the PID of the most recently backgrounded process. Write it down -- you will need it to check on the job later.

In [ ]:
# nohup keeps a job running even after you close the terminal:
#   nohup python train.py --epochs 100 > training.log 2>&1 &
#
# This is essential for long training runs on remote machines.
# Again, we run synchronously here for the notebook:

!python train.py --lr 0.001 --epochs 3 --output-dir /tmp/experiments/nohup_run > /tmp/experiments/nohup.log 2>&1
!echo "=== nohup job log ==="
!cat /tmp/experiments/nohup.log

In [ ]:
# In a real terminal session, you would use:
#   jobs          # list background jobs
#   wait $PID     # wait for a specific job to finish
#   kill $PID     # stop a job
#
# These commands work in the Colab terminal but not in notebook cells.
!echo "Background process management is a terminal skill — practise in Colab terminal."

### `wait` -- wait for background jobs

If you start multiple background jobs and need to wait for all of them to finish before proceeding, `wait` blocks until they complete.

```bash
python train.py --lr 0.001 > run1.log 2>&1 &
python train.py --lr 0.01  > run2.log 2>&1 &
python train.py --lr 0.1   > run3.log 2>&1 &

wait  # blocks until all three finish
echo "All runs complete."
```

### Exercise 6: Background training

**Try this:** Start a training run in the background with `nohup`, wait for it to finish, then display the log.

In [ ]:
%%writefile bg_exercise.sh
#!/bin/bash
# This script demonstrates background processes.
# Run it in the Colab terminal, not in a notebook cell.

echo "Starting training job..."
python train.py --lr 0.01 --epochs 3 --output-dir /tmp/experiments/bg_ex > /tmp/experiments/bg_exercise.log 2>&1 &
BG_PID=$!
echo "Background PID: $BG_PID"

echo "Waiting for job to finish..."
wait $BG_PID
echo "Job complete. Final result:"
tail -1 /tmp/experiments/bg_exercise.log

In [ ]:
!chmod +x bg_exercise.sh
!./bg_exercise.sh

---

## 8. Putting It Together -- A Hyperparameter Sweep Script

This is the culmination of everything we have learned: variables, conditionals, loops, logging, and the training script -- combined into a single sweep that tries multiple configurations and summarises the results.

Before tools like Weights & Biases existed, this is how people tracked experiments -- and many still do for quick explorations. Understanding this pattern will also make you appreciate *why* tools like W&B were invented.

In [ ]:
%%writefile sweep.sh
#!/bin/bash
# Hyperparameter sweep: try different learning rates and optimisers

RESULTS_DIR="sweep_results_$(date +%Y%m%d)"
mkdir -p "$RESULTS_DIR"

echo "============================================"
echo "  Hyperparameter Sweep"
echo "============================================"
echo "Results will be saved to: $RESULTS_DIR"
echo ""

RUN_COUNT=0

for LR in 0.001 0.01 0.1; do
    for OPT in sgd adam; do
        RUN_COUNT=$((RUN_COUNT + 1))
        RUN_NAME="${OPT}_lr${LR}"
        LOG_FILE="${RESULTS_DIR}/${RUN_NAME}.log"

        echo "--- Run $RUN_COUNT: $RUN_NAME ---"
        python train.py --lr "$LR" --optimizer "$OPT" --epochs 5 \
            --output-dir "${RESULTS_DIR}/${RUN_NAME}" > "$LOG_FILE" 2>&1

        # Extract the final epoch line from the log
        FINAL_LINE=$(grep "Epoch 5/5" "$LOG_FILE")
        echo "  Result: $FINAL_LINE"
    done
done

echo ""
echo "============================================"
echo "  Sweep complete: $RUN_COUNT runs"
echo "============================================"
echo ""
echo "Files in $RESULTS_DIR:"
ls "$RESULTS_DIR"/*.log

echo ""
echo "=== Top 3 results by accuracy ==="
grep -h "Epoch 5/5" "$RESULTS_DIR"/*.log | sort -t: -k3 -rn | head -3

In [ ]:
!chmod +x sweep.sh
!./sweep.sh

Let us look at one of the log files in detail to see exactly what was captured.

In [ ]:
!echo "=== Contents of adam_lr0.01 log ==="
!cat sweep_results_*/adam_lr0.01.log

And the JSON metrics file that `train.py` saved alongside the log.

In [ ]:
!cat sweep_results_*/adam_lr0.01/metrics.json

---

## 9. Quick Reference

A summary of every scripting concept covered in this session.

### Script basics

| Concept | Syntax | Example |
|---|---|---|
| Shebang | `#!/bin/bash` | First line of every script |
| Make executable | `chmod +x script.sh` | `chmod +x train.sh` |
| Run a script | `./script.sh` | `./train.sh 0.01 10` |
| Arguments | `$1`, `$2`, `$#`, `$@` | `LR=$1` |
| Default values | `${VAR:-default}` | `LR=${1:-0.001}` |

### Variables and arithmetic

| Concept | Syntax | Example |
|---|---|---|
| String variable | `VAR="value"` | `MODEL="resnet"` |
| Arithmetic | `$((expression))` | `TOTAL=$((EPOCHS * BATCHES))` |
| Command substitution | `$(command)` | `DATE=$(date +%Y-%m-%d)` |
| String interpolation | `"${VAR}_suffix"` | `"runs/${MODEL}_${DATE}"` |

### Conditionals

| Test | Meaning |
|---|---|
| `[ -f file ]` | File exists |
| `[ -d dir ]` | Directory exists |
| `[ -s file ]` | File is non-empty |
| `[ "$A" = "$B" ]` | String equality |
| `[ $N -gt 5 ]` | Greater than |
| `[ $N -lt 5 ]` | Less than |
| `[ $N -eq 5 ]` | Equal |

### Loops

| Type | Syntax |
|---|---|
| For (list) | `for X in a b c; do ... done` |
| For (files) | `for F in *.csv; do ... done` |
| While | `while [ condition ]; do ... done` |

### Output and logging

| Pattern | Syntax | Effect |
|---|---|---|
| Redirect stdout | `command > file` | Output to file (overwrite) |
| Append | `command >> file` | Output to file (append) |
| Redirect all | `command > file 2>&1` | stdout + stderr to file |
| Tee | `command \| tee file` | Screen and file simultaneously |
| Watch log | `tail -f file` | Follow log in real time |

### Background processes

| Pattern | Syntax | Effect |
|---|---|---|
| Background | `command &` | Run without blocking |
| Survive disconnect | `nohup command &` | Keep running after logout |
| Get PID | `$!` | PID of last backgrounded job |
| Wait | `wait` | Block until all background jobs finish |

---

## Summary

Bash scripts automate the repetitive parts of ML workflows. The core pattern is always the same:

1. **Parse arguments** -- accept hyperparameters and paths from the command line
2. **Check preconditions** -- verify data exists, directories are ready, dependencies are installed
3. **Run training** -- call the Python script with the right arguments
4. **Log output** -- capture everything to a file for later review

Hyperparameter sweeps are just nested `for` loops in a script. Before dedicated experiment tracking tools existed, this was how every ML researcher compared configurations -- and the technique remains useful for quick explorations.

You now have the tools to run training jobs on any Linux machine -- your laptop, Colab, a university cluster, or a cloud GPU. The commands are the same everywhere.

---

## Cleanup

Remove all files and directories created during this session.

In [ ]:
!rm -f hello.sh greet.sh greet_user.sh personal_greet.sh
!rm -f config_demo.sh arithmetic_demo.sh substitution_demo.sh setup_experiment.sh
!rm -f check_files.sh check_args.sh preflight.sh check_ready.sh
!rm -f loop_demo.sh lr_sweep.sh while_demo.sh nested_sweep.sh lr_loop.sh
!rm -f train.py run_experiment.sh sweep.sh bg_exercise.sh
!rm -f config.yaml nohup.out
!rm -rf data/ results/ sweep_results_*/
!rm -rf /tmp/experiments /tmp/test_run
!echo "Cleaned up all session files."